<a href="https://colab.research.google.com/github/aleja71291/FDL-EA-20252/blob/main/1_Exploracion_de_datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introducción
La enfermedad de Alzheimer es una afección neurodegenerativa, con bases moleculares complejas que culminan en muerte neuronal, reflejado en atrofia cerebral. Los procesos moleculares patológicos en la enfermedad tienen una distribución espacial característica de la enfermedad, con afectación predominante de estructuras como los lóbulos temporales mediales y giros fusiformes en fases tempranas, y generalización del acúmulo de proteinas patogénicas en fases más tardías. El grado y distribución de la atrofia permiten su correlación con el estadío clínico de la enfermedad, sin embargo sel comportamiento de estas variables no es lineal, por lo que el uso de estrategias estadísticas tradicionales se encuentra limitado. Por lo anterior, cada vez es más prevalente el uso de modelos de IA para esta tarea, ya que permite encontrar correlaciones entre parámetros y patrones de comportamiento de variables que pueden no ser evidentes a través de análisis tradicionales.

# Descripción y objetivo del modelo
Se pretende entrenar un modelo basado en redes convolucionales para clasificar el estadio de la enfermedad de Alzheimer, a partir de datos de análisis estructural de imágenes de resonancia magnética cerebral.

# Criterios de clasificación:
Como criterio de clasificación del modelo, se utilizará el resultado de la escala clobal de CDR (clinical dementia rating).
- 0 = Sin déficit cognitivo
- 0.5 = déficit cognitivo leve
- 1.0 = demencia leve
- 2.0 = demencia moderada
- 3.0 = demencia severa

In [1]:
import pandas as pd
csv_url = "https://raw.githubusercontent.com/aleja71291/FDL-EA-20252/main/content/CDR_13Oct2025.csv"
CDR = pd.read_csv(csv_url)
print(CDR.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14576 entries, 0 to 14575
Data columns (total 25 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   PHASE                 14576 non-null  object 
 1   PTID                  14576 non-null  object 
 2   RID                   14576 non-null  int64  
 3   VISCODE               14576 non-null  object 
 4   VISCODE2              14568 non-null  object 
 5   VISDATE               14569 non-null  object 
 6   CDSOURCE              14399 non-null  float64
 7   CDVERSION             9575 non-null   float64
 8   SPID                  4420 non-null   float64
 9   CDMEMORY              14468 non-null  float64
 10  CDORIENT              14469 non-null  float64
 11  CDJUDGE               14469 non-null  float64
 12  CDCOMMUN              14468 non-null  float64
 13  CDHOME                14468 non-null  float64
 14  CDCARE                14468 non-null  float64
 15  CDGLOBAL           

# Resonancia magnética estructural
Como parámetros para la clasificación, se utilizarán los volúmnes de estructuras cerebrales, calculados a través de Freesurfer, versión 7.x. En la siguiente base de datos se encuentran los resultados del análisis estructural de la cohorte ADNI1.

In [2]:
import pandas as pd
csv_url = "https://raw.githubusercontent.com/aleja71291/FDL-EA-20252/refs/heads/main/content/UCSFFSX7_ADNI1.csv"
ADNI1 = pd.read_csv(csv_url, delimiter=';')

# Preparación de los datos

Combinamos ambos datasets, basándonos en las columnas comunes "PTID" y "VISCODE". Antes de combinar ambos datasets, se armoniza la codificación de VISCODE.
Ambos datasets contienen información de múltiples visitas de cada participante. por el momento nos interesa obtener información transversal, por lo que exploraremos la distribución de clases en las diferentes visitas.

In [3]:
CDR['VISCODE'] = CDR['VISCODE'].replace('f', 'bl')
CDR['VISCODE'] = CDR['VISCODE'].replace('sc', 'bl')
CDR_bl = CDR[CDR['VISCODE'] == 'bl']
ADNI1['VISCODE'] = ADNI1['VISCODE'].replace('sc', 'bl')
ADNI1_bl = ADNI1[ADNI1['VISCODE'] == 'bl']
ADNI1_bl15 = ADNI1_bl[ADNI1_bl['FIELD_STRENGTH'] == '1.5T']
merged_df = pd.merge(CDR_bl, ADNI1_bl15, on=['PTID'], how='inner')
print(merged_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 837 entries, 0 to 836
Columns: 371 entries, PHASE_x to update_stamp_y
dtypes: float64(208), int64(144), object(19)
memory usage: 2.4+ MB
None


In [4]:
distribution = merged_df['CDGLOBAL'].value_counts()
print(distribution)

CDGLOBAL
0.5    503
0.0    232
1.0    102
Name: count, dtype: int64


Tomando únicamente las visitas iniciales de cada participante, no contamos con observaciones de las clases CDR 2 ni 3. Revisaremos la distribución de clases en las demás visitas. Visita de 6 meses:

In [5]:
CDR_m06 = CDR[CDR['VISCODE'] == 'm06']
ADNI1_m06 = ADNI1[ADNI1['VISCODE'] == 'm06']
ADNI1_m0615 = ADNI1_m06[ADNI1_m06['FIELD_STRENGTH'] == '1.5T']
merged_dfm06 = pd.merge(CDR_m06, ADNI1_m0615, on=['PTID'], how='inner')
print(merged_dfm06.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 740 entries, 0 to 739
Columns: 371 entries, PHASE_x to update_stamp_y
dtypes: float64(208), int64(144), object(19)
memory usage: 2.1+ MB
None


In [6]:
distribution = merged_dfm06['CDGLOBAL'].value_counts()
print(distribution)

CDGLOBAL
 0.5    404
 0.0    216
 1.0    107
 2.0     10
-1.0      3
Name: count, dtype: int64


Visita de 12 meses

In [7]:
CDR_m12 = CDR[CDR['VISCODE'] == 'm12']
ADNI1_m12 = ADNI1[ADNI1['VISCODE'] == 'm12']
ADNI1_m1215 = ADNI1_m12[ADNI1_m12['FIELD_STRENGTH'] == '1.5T']
merged_dfm12 = pd.merge(CDR_m12, ADNI1_m1215, on=['PTID'], how='inner')
print(merged_dfm12.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 687 entries, 0 to 686
Columns: 371 entries, PHASE_x to update_stamp_y
dtypes: float64(208), int64(144), object(19)
memory usage: 1.9+ MB
None


In [8]:
distribution = merged_dfm12['CDGLOBAL'].value_counts()
print(distribution)

CDGLOBAL
 0.5    354
 0.0    194
 1.0    112
 2.0     18
-1.0      7
 3.0      2
Name: count, dtype: int64


Visita de 18 meses

In [9]:
CDR_m18 = CDR[CDR['VISCODE'] == 'm18']
ADNI1_m18 = ADNI1[ADNI1['VISCODE'] == 'm18']
ADNI1_m1815 = ADNI1_m18[ADNI1_m18['FIELD_STRENGTH'] == '1.5T']
merged_dfm18 = pd.merge(CDR_m18, ADNI1_m1815, on=['PTID'], how='inner')
print(merged_dfm18.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 290 entries, 0 to 289
Columns: 371 entries, PHASE_x to update_stamp_y
dtypes: float64(208), int64(144), object(19)
memory usage: 840.7+ KB
None


In [10]:
distribution = merged_dfm18['CDGLOBAL'].value_counts()
print(distribution)

CDGLOBAL
 0.5    223
 1.0     52
 0.0     12
-1.0      2
 3.0      1
Name: count, dtype: int64


Visita 24 meses

In [11]:
CDR_m24 = CDR[CDR['VISCODE'] == 'm24']
ADNI1_m24 = ADNI1[ADNI1['VISCODE'] == 'm24']
ADNI1_m2415 = ADNI1_m24[ADNI1_m24['FIELD_STRENGTH'] == '1.5T']
merged_dfm24 = pd.merge(CDR_m24, ADNI1_m2415, on=['PTID'], how='inner')
print(merged_dfm24.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 560 entries, 0 to 559
Columns: 371 entries, PHASE_x to update_stamp_y
dtypes: float64(208), int64(144), object(19)
memory usage: 1.6+ MB
None


In [12]:
distribution = merged_dfm24['CDGLOBAL'].value_counts()
print(distribution)

CDGLOBAL
 0.5    230
 0.0    176
 1.0    105
 2.0     37
-1.0      9
 3.0      3
Name: count, dtype: int64


Visita 36 meses

In [13]:
CDR_m36 = CDR[CDR['VISCODE'] == 'm36']
ADNI1_m36 = ADNI1[ADNI1['VISCODE'] == 'm36']
ADNI1_m3615 = ADNI1_m36[ADNI1_m36['FIELD_STRENGTH'] == '1.5T']
merged_dfm36 = pd.merge(CDR_m36, ADNI1_m3615, on=['PTID'], how='inner')
print(merged_dfm36.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 355 entries, 0 to 354
Columns: 371 entries, PHASE_x to update_stamp_y
dtypes: float64(208), int64(144), object(19)
memory usage: 1.0+ MB
None


In [14]:
distribution = merged_dfm36['CDGLOBAL'].value_counts()
print(distribution)

CDGLOBAL
 0.5    140
 0.0    138
 1.0     63
 2.0     10
-1.0      3
 3.0      1
Name: count, dtype: int64


Existe un disbalance de clases cuando se toman las visitas como criterio de selección de la muestra. En todas las visitas, la clase de demencia severa (CDR 3) se encuentra subrepresentada significativamente, y en menor medida la clase de demencia moderada (CDR 2). El valor -1 no existe en la escala CDR, por lo que se considera un valor faltante. Con el propósito de alcanzar un mejor balance de clases, fusionamos las clases de demencia moderada y severa (CDR 2 y CDR 3), y utilizamos una estrategia de balanceo de clases (SMOTE) de sobremuestreo (posterior a subdivisión en sets de entrenamiento y validación). Utilizamos los datos de la visita de 24 meses, ya que es en la que la clase CDR 2-3 representa un porcentaje cercano al 10% de la muestra.

In [15]:
merged_dfm24 = merged_dfm24[merged_dfm24['CDGLOBAL'] != -1]
merged_dfm24['CDGLOBAL'] = merged_dfm24['CDGLOBAL'].replace({0.0: 0, 0.5: 1, 1.0: 2, 2.0: 3, 3.0: 3})
distribution = merged_dfm24['CDGLOBAL'].value_counts()
print(distribution)

CDGLOBAL
1.0    230
0.0    176
2.0    105
3.0     40
Name: count, dtype: int64


In [16]:
import pandas as pd
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.impute import SimpleImputer

merged_dfm24['CDGLOBAL'] = merged_dfm24['CDGLOBAL'].astype(int)

# Separar las variables de interes "X" y "y"
X = merged_dfm24.drop('CDGLOBAL', axis=1)
y = merged_dfm24['CDGLOBAL']

# Limpieza de datos
non_numeric_cols = X.select_dtypes(exclude=np.number).columns
X = X.drop(columns=non_numeric_cols)
cols_with_all_nans = X.columns[X.isnull().all()]
X = X.drop(cols_with_all_nans, axis=1)

# Separar los datos en set de entrenamiento y set de prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

if isinstance(X_train, pd.DataFrame):
    X_train.columns = X.columns
if isinstance(X_test, pd.DataFrame):
    X_test.columns = X.columns

# Imputar valores faltantes en el set de entrenamiento
imputer = SimpleImputer(strategy='mean')
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

# Aplicar técnica de remuestreo (SMOTE) en el set de entrenamiento
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train, y_train)

# Convertir los datos remuestreados a DataFrame de pandas
X_resampled_df = pd.DataFrame(X_resampled, columns=X.columns)
y_resampled_df = pd.Series(y_resampled, name='CDGLOBAL')

merged_dfm24_resampled = pd.concat([X_resampled_df, y_resampled_df], axis=1)

print("Original training data class distribution:")
print(y_train.value_counts())

print("\nResampled training data class distribution:")
print(y_resampled_df.value_counts())


Original training data class distribution:
CDGLOBAL
1    184
0    140
2     84
3     32
Name: count, dtype: int64

Resampled training data class distribution:
CDGLOBAL
1    184
0    184
2    184
3    184
Name: count, dtype: int64


Nota: no es necesario descargar los datos resultantes. Los demás cuadernos los toman de GitHub.

In [17]:
merged_dfm24.to_csv('merged_dfm24.csv', index=False)
merged_dfm24_resampled.to_csv('merged_dfm24_resampled.csv', index=False)
X_resampled_df.to_csv('X_resampled_df.csv', index=False)
y_resampled_df.to_csv('y_resampled_df.csv', index=False)
df_X_test = pd.DataFrame(X_test)
df_X_test.to_csv('X_test.csv', index=False)
df_y_test = pd.DataFrame(y_test)
df_y_test.to_csv('y_test.csv', index=False)

from google.colab import files
files.download('merged_dfm24.csv')
files.download('merged_dfm24_resampled.csv')
files.download('X_resampled_df.csv')
files.download('y_resampled_df.csv')
files.download('X_test.csv')
files.download('y_test.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>